In [6]:
import os
import time
global notebook
notebook = 1

dir = '/global/u1/j/jackh/bh/harm2d'
os.chdir(dir)

%run -i setup.py build_ext --inplace
%run -i pp.py build_ext --inplace
%matplotlib inline
import matplotlib
print('Imports done.')

Imports done.


In [7]:
global notebook, axisym,set_cart,axisym,REF_1,REF_2,REF_3,set_cart,D,print_fieldlines
global lowres1,lowres2,lowres3, RAD_M1, RESISTIVE, export_raytracing_GRTRANS, export_raytracing_RAZIEH,r1,r2,r3
global r_min, r_max, theta_min, theta_max, phi_min,phi_max, do_griddata, do_box, check_files, kerr_schild

dir = '/pscratch/sd/l/lalakos/ml_data_rc300/reduced'
os.chdir(dir)

# set params
lowres1 = 1
lowres2 = 1
lowres3 = 1

do_box=0
r_min=1.0
r_max=100.0
theta_min=0.0
theta_max=9
phi_min=-1
phi_max=9
axisym=1
print_fieldlines=0
export_raytracing_GRTRANS=0
export_raytracing_RAZIEH=0
kerr_schild=0
DISK_THICKNESS=0.03
set_cart=0
set_mpi(0)
check_files=1
notebook=1
interpolate_var=0
AMR = 0 # get all data in grid

In [3]:
t

NameError: name 't' is not defined

In [8]:
dumps_path = '/pscratch/sd/l/lalakos/ml_data_rc300/reduced'
start_dump = 3000
num_dumps = 4
rblock_new_ml()
for dump in np.random():
    rpar_new(dump)
    if dump == start_dump: rgdump_griddata(dumps_path)
    rdump_griddata(dumps_path, dump)

    arr = np.concatenate((rho, ug, np.squeeze(uu[1:4], axis=1), np.squeeze(B[1:4], axis=1)), axis=0)
    print(arr.shape)


(8, 224, 48, 96)
(8, 224, 48, 96)
(8, 224, 48, 96)
(8, 224, 48, 96)


In [17]:
import os
import numpy as np
import h5py

DUMPS_DIR = '/pscratch/sd/l/lalakos/ml_data_rc300/reduced'
DATA_PATH = os.getenv('SCRATCH')+"/data.hdf5"

# populate HDF5 file with data from source dumps
def populate_h5(dumps_path:str, start_dump: int, num_dumps: int):
    with h5py.File(DATA_PATH, "a") as f:
        rblock_new_ml()
        for dump in range(start_dump, start_dump+num_dumps):
            rpar_new(dump)
            if dump == start_dump:
                rgdump_griddata(dumps_path)
            rdump_griddata(dumps_path, dump)

            # construct array
            new_data = np.expand_dims(np.concatenate((np.log(rho), ug, np.squeeze(uu[1:4], axis=1), np.squeeze(B[1:4], axis=1)), axis=0),0)
            new_label = np.array([[dump]])
            
            if dump == start_dump:
                f.create_dataset('data', data=new_data, compression="gzip", chunks=True, maxshape=(None,8,224,48,96))
                f.create_dataset('dump_index', data=new_label, compression="gzip", chunks=True, maxshape=(None,1))
            else:
                f['data'].resize((f['data'].shape[0] + new_data.shape[0]), axis=0)
                f['data'][-new_data.shape[0]:] = new_data
            
                f['dump_index'].resize((f['dump_index'].shape[0] + new_label.shape[0]), axis=0)
                f['dump_index'][-new_label.shape[0]:] = new_label

            if (dump % 50 == 0): print(f'Dump {dump} written.')

start_dump = 3000
num_dumps = 4000
populate_h5(DUMPS_DIR, start_dump=start_dump, num_dumps=num_dumps)

Dump 3000 written.
Dump 3050 written.
Dump 3150 written.
Dump 3200 written.
Dump 3300 written.
Dump 3350 written.
Dump 3400 written.
Dump 3450 written.
Dump 3500 written.
Dump 3550 written.
Dump 3600 written.
Dump 3650 written.
Dump 3700 written.
Dump 3750 written.
Dump 3800 written.
Dump 3850 written.
Dump 3900 written.
Dump 3950 written.
Dump 4000 written.
Dump 4050 written.
Dump 4100 written.
Dump 4150 written.
Dump 4200 written.
Dump 4250 written.
Dump 4300 written.
Dump 4350 written.
Dump 4400 written.
Dump 4450 written.
Dump 4500 written.
Dump 4550 written.
Dump 4600 written.
Dump 4650 written.
Dump 4700 written.
Dump 4750 written.
Dump 4800 written.
Dump 4850 written.
Dump 4900 written.
Dump 4950 written.
Dump 5000 written.
Dump 5050 written.
Dump 5100 written.
Dump 5150 written.
Dump 5200 written.
Dump 5250 written.
Dump 5300 written.
Dump 5350 written.
Dump 5400 written.
Dump 5450 written.
Dump 5500 written.
Dump 5550 written.
Dump 5600 written.
Dump 5650 written.
Dump 5700 wr

In [31]:
import matplotlib.pyplot as plt
from time import time

with h5py.File(DATA_PATH, "r") as f:
    for idx in [20,400,812,3001,2,4,71,2048]:
        start = time()
        var = f['data'][idx][0,:,:,:]
        print(f'Im fetched in {time()-start:.4f}s')

        # plt.imshow(var)
        # plt.axis('off')
        # plt.show()
    

Im fetched in 0.4905s
Im fetched in 0.4235s
Im fetched in 0.5329s
Im fetched in 0.4372s
Im fetched in 0.3876s
Im fetched in 0.3243s
Im fetched in 0.4118s
Im fetched in 0.6760s


In [8]:
import datasets
import os

def dumps_generator(dumps_path:str, start_dump: int, num_dumps: int):
    rblock_new_ml()
    for dump in range(start_dump, start_dump+num_dumps):
        rpar_new(dump)
        if dump == start_dump: rgdump_griddata(dumps_path)
        rdump_griddata(dumps_path, dump)

        arr = np.concatenate((rho, ug, np.squeeze(uu[1:4], axis=1), np.squeeze(B[1:4], axis=1)), axis=0)
        
        yield {'dump': dump, 'time': t, 'frame': arr}

DATA_DIR = '/pscratch/sd/l/lalakos/ml_data_rc300/reduced'
CACHE_DIR = os.getenv('SCRATCH')+'/cache'
SAVE_DIR = os.getenv('SCRATCH')+'/data'
start_dump = 3000
num_dumps = 500
hf_dataset = datasets.Dataset.from_generator(
    dumps_generator, 
    gen_kwargs ={"dumps_path": DATA_DIR, "start_dump": start_dump, "num_dumps": num_dumps},
    cache_dir = CACHE_DIR
)

print(f'Writing dataset\n{hf_dataset}')
hf_dataset.save_to_disk(SAVE_DIR)
hf_dataset.cleanup_cache_files()

Generating train split: 0 examples [00:00, ? examples/s]

DatasetGenerationError: An error occurred while generating the dataset